## Run SpaceFlow

In [1]:
# Python env: SpaceFlow
from numpy.random import default_rng
import scanpy as sc
# import squidpy as sq
from anndata import AnnData
import scipy
sc.logging.print_header()
from sklearn.metrics.cluster import adjusted_rand_score
import numpy as np
import pandas as pd
import seaborn as sns
import os
import torch
import pandas as pd
from sklearn import metrics
import multiprocessing as mp
from sklearn.metrics.cluster import normalized_mutual_info_score
from SpaceFlow import SpaceFlow

/home/vanhoan310/miniconda3/envs/spaceflow/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scanpy==1.8.1 anndata==0.8.0 numpy==1.21.6 scipy==1.7.3 pandas==1.3.5 scikit-learn==1.0.2 statsmodels==0.13.5 python-igraph==0.10.8 pynndescent==0.5.13


/home/vanhoan310/miniconda3/envs/spaceflow/lib/python3.7/site-packages/torch_geometric/typing.py:31: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /data/hoan/mybin/miniconda3/envs/spaceflow/lib/python3.7/site-packages/torch_scatter/_version_cpu.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/home/vanhoan310/miniconda3/envs/spaceflow/lib/python3.7/site-packages/torch_geometric/typing.py:42: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /data/hoan/mybin/miniconda3/envs/spaceflow/lib/python3.7/site-packages/torch_sparse/_version_cpu.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


In [2]:
adata = sc.read_visium('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data', count_file='filtered_feature_bc_matrix.h5')
ground_truth = pd.read_csv('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data/metadata.tsv', delimiter='\t', index_col=0)
adata.obs['ground_truth'] = ground_truth['ground_truth']
adata.var_names_make_unique()

/home/vanhoan310/miniconda3/envs/spaceflow/lib/python3.7/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [3]:
data_name = 'SpaceFlowBreastCancer'
sc.pp.filter_genes(adata, min_cells=1)
sf = SpaceFlow.SpaceFlow(adata=adata)
sf.preprocessing_data(n_top_genes=10)
embed_filepath = 'output/SpaceFlow_BreastCancer_embedding.tsv'
domain_label = 'output/SpaceFlow_BreastCancer_domains.tsv'
sf.train(spatial_regularization_strength=0.1, z_dim=50, lr=1e-3, epochs=1000, max_patience=50, min_stop=100, random_seed=0, gpu=0, regularization_acceleration=True, edge_subset_sz=1000000)

Epoch 2/1000, Loss: 1.4278242588043213
Epoch 12/1000, Loss: 1.276574730873108
Epoch 22/1000, Loss: 1.112086296081543
Epoch 32/1000, Loss: 0.945341944694519
Epoch 42/1000, Loss: 0.7773937582969666
Epoch 52/1000, Loss: 0.6729124188423157
Epoch 62/1000, Loss: 0.582268476486206
Epoch 72/1000, Loss: 0.5510807037353516
Epoch 82/1000, Loss: 0.5208783149719238
Epoch 92/1000, Loss: 0.5070530772209167
Epoch 102/1000, Loss: 0.4933009445667267
Epoch 112/1000, Loss: 0.4728069007396698
Epoch 122/1000, Loss: 0.4687155485153198
Epoch 132/1000, Loss: 0.4887300133705139
Epoch 142/1000, Loss: 0.4249113202095032
Epoch 152/1000, Loss: 0.4200304448604584
Epoch 162/1000, Loss: 0.41551247239112854
Epoch 172/1000, Loss: 0.3850571811199188
Epoch 182/1000, Loss: 0.4146420359611511
Epoch 192/1000, Loss: 0.3915235698223114
Epoch 202/1000, Loss: 0.36190661787986755
Epoch 212/1000, Loss: 0.3668627440929413
Epoch 222/1000, Loss: 0.3485269248485565
Epoch 232/1000, Loss: 0.38641127943992615
Epoch 242/1000, Loss: 0.3317

array([[-1.3380136e-05, -1.1521336e-02,  2.9815570e-02, ...,
         2.1939293e-01,  1.9738336e-01,  2.5625704e-02],
       [ 2.9749137e-01,  1.4407200e-01,  1.6902436e-01, ...,
        -2.4486063e-02, -4.6858713e-02,  2.8814882e-01],
       [-2.4154683e-04, -3.3381782e-02,  2.4707353e-01, ...,
         7.6272741e-02,  1.5978027e-02,  1.8533710e-02],
       ...,
       [ 3.2366741e-01,  1.5957057e-01,  1.4679401e-01, ...,
        -3.0091792e-02, -5.1631156e-02,  3.4580737e-01],
       [-5.2648675e-06, -1.1362931e-02,  1.8363659e-01, ...,
         2.5188065e-01,  1.2011872e-01,  1.5146632e-02],
       [-1.7582670e-04, -3.7440274e-02,  1.4374636e-02, ...,
         2.2108313e-01, -3.4112703e-02,  7.5770230e-03]], dtype=float32)

In [4]:
sf.segmentation(domain_label_save_filepath=domain_label, n_neighbors=50, resolution=1.473)
df_out = pd.read_csv('output/SpaceFlow_BreastCancer_domains.tsv',header=None)
SpaceFlow_labels = df_out.values.flatten()
adata.obs['SpaceFlow'] = SpaceFlow_labels

true_labels = pd.factorize(adata.obs['ground_truth'])[0] 
n_clusters = len(np.unique(true_labels))
print(len(np.unique(SpaceFlow_labels)), len(np.unique(true_labels)))

Performing domain segmentation
Segmentation complete, domain labels of cells or spots saved at output/SpaceFlow_BreastCancer_domains.tsv !
20 20


In [5]:
NMI = normalized_mutual_info_score(adata.obs['SpaceFlow'], adata.obs['ground_truth'])
print(NMI)

0.537929705172982


In [6]:
adata.write_h5ad(
    '/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data/Section_1_SpaceFlow.h5ad'
)